# Virtual Memory Simulator - Experiments and Analysis

This notebook demonstrates batch runs, pandas summaries, and plots for the Virtual Memory Management Tool.

In [ ]:
import sys
import os
from pathlib import Path
import json

# Add simulator directory to path
sys.path.insert(0, str(Path.cwd().parent / 'simulator'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Import simulator modules
from simulator.core import Simulator
from simulator.algorithms import get_algorithm
from simulator.io import load_scenario
from simulator.metrics import MetricsCollector, analyze_timeline, compare_algorithms

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

In [ ]:
# Save to CSV
output_path = '../results.csv'
df.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")

## 6. Save Results to CSV

In [ ]:
# Create pivot table for heatmap
pivot_faults = df.pivot(index='algorithm', columns='frames', values='page_faults')

plt.figure(figsize=(10, 6))
sns.heatmap(pivot_faults, annot=True, fmt='g', cmap='YlOrRd', cbar_kws={'label': 'Page Faults'})
plt.title('Page Faults Heatmap: Algorithm vs Frame Count')
plt.ylabel('Algorithm')
plt.xlabel('Number of Frames')
plt.show()

## 5. Heatmap Visualization

In [ ]:
# Summary statistics by algorithm
summary = df.groupby('algorithm').agg({
    'page_faults': ['mean', 'min', 'max'],
    'hit_ratio': ['mean', 'min', 'max']
}).round(4)

print("Algorithm Performance Summary:")
print(summary)

# Find best algorithm
best_algo = df.groupby('algorithm')['page_faults'].mean().idxmin()
print(f"\nBest overall algorithm: {best_algo}")

## 4. Algorithm Comparison Summary

In [ ]:
# Plot page faults vs frame count
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

for algo in algorithms:
    algo_data = df[df['algorithm'] == algo]
    ax1.plot(algo_data['frames'], algo_data['page_faults'], marker='o', label=algo)

ax1.set_xlabel('Number of Frames')
ax1.set_ylabel('Page Faults')
ax1.set_title('Page Faults vs Frame Count')
ax1.legend()
ax1.grid(True)

# Plot hit ratio vs frame count
for algo in algorithms:
    algo_data = df[df['algorithm'] == algo]
    ax2.plot(algo_data['frames'], algo_data['hit_ratio']*100, marker='s', label=algo)

ax2.set_xlabel('Number of Frames')
ax2.set_ylabel('Hit Ratio (%)')
ax2.set_title('Hit Ratio vs Frame Count')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 3. Visualize Results

In [ ]:
# Load thrashing scenario
scenario_path = '../examples/scenarios/thrashing.json'
scenario = load_scenario(scenario_path)

# Run batch experiments
frame_counts = [2, 3, 4, 5, 6, 7, 8]
algorithms = ['FIFO', 'LRU', 'CLOCK', 'Optimal']

results = []
for frames in frame_counts:
    for algo_name in algorithms:
        algorithm = get_algorithm(algo_name)
        simulator = Simulator(num_frames=frames, algorithm=algorithm)
        result = simulator.run(scenario['accesses'])
        
        results.append({
            'frames': frames,
            'algorithm': algo_name,
            'page_faults': result['metrics']['page_faults'],
            'hits': result['metrics']['hits'],
            'hit_ratio': result['metrics']['hit_ratio'],
            'avg_access_time_ms': result['metrics']['avg_access_time_ms']
        })

# Create DataFrame
df = pd.DataFrame(results)
print(df.head(10))

## 2. Run Batch Experiments

## 1. Import Required Libraries